# Day 2 - Lab 3: Feature engineering, framing and communicating

**Goal:** turn the cleaned table into a **model-ready** feature table for Day 3, then frame the business question and write the finding in plain English.

Feature engineering is where domain knowledge enters. A model is only as good as the columns you give it.

## 1. Load

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()

def _rebuild_clean():
    """Re-run the Day 1 cleaning from raw, so Day 2 is self-contained."""
    df = pd.read_csv(DATA / 'raw' / 'service_requests_raw.csv', dtype=str).drop_duplicates()
    df['priority'] = df['priority'].str.strip().str.title().replace({'2': 'Medium'})
    df['resolution_hours'] = pd.to_numeric(df['resolution_hours'], errors='coerce')
    iso   = pd.to_datetime(df['submitted_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    named = pd.to_datetime(df['submitted_at'], format='%d-%b-%Y %H:%M', errors='coerce')
    df['submitted_at'] = iso.fillna(named)
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.loc[(df['resolution_hours'] < 0) | (df['resolution_hours'] > 8760), 'resolution_hours'] = np.nan
    df.loc[(df['resolved_at'] < df['submitted_at']).fillna(False), 'resolved_at'] = pd.NaT
    df['citizen_age_band'] = df['citizen_age_band'].replace({'': pd.NA, 'Unknown': pd.NA}).fillna('Unknown')
    df['service_id'] = pd.to_numeric(df['service_id'], errors='coerce')
    svc = pd.read_csv(DATA / 'seeds' / 'services.csv')
    df = df.merge(svc[['service_id', 'target_resolution_hours']], on='service_id', how='left')
    resolved = df['status'].isin(['Resolved', 'Reopened']) & df['resolution_hours'].notna()
    df['sla_met'] = pd.NA
    df.loc[resolved, 'sla_met'] = (df.loc[resolved, 'resolution_hours'] <= df.loc[resolved, 'target_resolution_hours']).astype('int')
    return df

def load_clean():
    p = DATA / 'processed' / 'service_requests_clean.csv'
    if p.exists():
        print('Loaded cleaned dataset from Day 1:', p.name)
        return pd.read_csv(p, parse_dates=['submitted_at', 'resolved_at'])
    print('Cleaned file not found, rebuilding from raw (Day 1 cleaning)...')
    return _rebuild_clean()

df = load_clean()
channels = pd.read_csv(DATA / 'seeds' / 'channels.csv')
districts = pd.read_csv(DATA / 'seeds' / 'districts.csv')
df = df.merge(channels, on='channel_id', how='left')
print('shape:', df.shape)
df.head(3)

Loaded cleaned dataset from Day 1: service_requests_clean.csv
shape: (12027, 16)


,request_id,submitted_at,service_id,channel_id,district_id,department_id,priority,status,resolved_at,resolution_hours,satisfaction_score,citizen_age_band,target_resolution_hours,sla_met,channel_name,is_digital
0,1003304,2024-07-13 06:38:00,13,1,101,2,Low,Open,NaT,99.2,NaN,35-44,72,NaN,Web Portal,True
1,1011864,2024-05-22 13:06:00,19,2,100,5,High,Resolved,2024-05-24 09:36:00,44.5,NaN,25-34,96,1.0,Mobile App,True
2,1001507,2024-02-13 01:07:00,10,1,104,1,Low,Resolved,2024-02-13 16:43:00,15.6,NaN,25-34,24,1.0,Web Portal,True


## 2. Temporal features
A single timestamp hides several useful signals. Pull them out: the hour of day, the day of week, the month, and whether it was a weekend.

In [2]:
df['submitted_hour']  = df['submitted_at'].dt.hour
df['submitted_dow']   = df['submitted_at'].dt.dayofweek        # 0 = Monday
df['submitted_month'] = df['submitted_at'].dt.month
df['is_weekend']      = df['submitted_dow'].isin([5, 6]).astype(int)
df[['submitted_at', 'submitted_hour', 'submitted_dow', 'is_weekend']].head()

,submitted_at,submitted_hour,submitted_dow,is_weekend
0,2024-07-13 06:38:00,6,5,1
1,2024-05-22 13:06:00,13,2,0
2,2024-02-13 01:07:00,1,1,0
3,2024-09-24 08:11:00,8,1,0
4,2024-07-08 07:16:00,7,0,0


In [3]:
print(f"weekend share: {df['is_weekend'].mean():.1%}")

weekend share: 28.6%


## 3. Flags and ratios
`is_digital` is already a flag. A more expressive feature is the **SLA ratio**: resolution time divided by the target. Above 1 means the target was missed, and by how much.

In [4]:
df['is_digital'] = df['is_digital'].astype(int)
df['sla_ratio']  = df['resolution_hours'] / df['target_resolution_hours']
print(f"median SLA ratio: {df['sla_ratio'].median():.2f}   (>1 means over target)")
print(f"share over target: {(df['sla_ratio'] > 1).mean():.1%}")

median SLA ratio: 0.93   (>1 means over target)
share over target: 44.7%


## 4. Join for context: district population
The district a request came from carries context. Join the population so the model can learn whether busier districts behave differently.

In [5]:
df = df.merge(districts[['district_id', 'population']], on='district_id', how='left')
print('population non-null:', df['population'].notna().sum(), 'of', len(df))

population non-null: 12027 of 12027


## 5. Encode categoricals
Models need numbers. `priority` is **ordinal** (Low < Medium < High), so map it to a rank. Unordered categories like status would use one-hot encoding (`pd.get_dummies`) instead.

In [6]:
df['priority_rank'] = df['priority'].map({'Low': 0, 'Medium': 1, 'High': 2})
df[['priority', 'priority_rank']].drop_duplicates()

,priority,priority_rank
0,Low,0
1,High,2
3,Medium,1


## 6. Assemble the model-ready table
Select the target and the features Day 3 will use, and save it. Keep the rows where the target is known for supervised learning.

In [7]:
features = [
    'sla_met',                 # target (classification)
    'resolution_hours',        # target (regression) / strong signal
    'submitted_hour', 'submitted_dow', 'submitted_month', 'is_weekend',
    'is_digital', 'population', 'priority_rank',
    'target_resolution_hours', 'satisfaction_score',
]
model_df = df[features].copy()
print('model-ready shape:', model_df.shape)
print('rows with known sla_met:', int(model_df['sla_met'].notna().sum()))

out = DATA / 'processed'
out.mkdir(parents=True, exist_ok=True)
model_df.to_csv(out / 'service_requests_features.csv', index=False)
print('saved ->', (out / 'service_requests_features.csv').name)

model-ready shape: (12027, 11)
rows with known sla_met: 10468
saved -> service_requests_features.csv


## 7. Frame the business problem
Before Day 3 touches a model, state the problem in one sentence a manager would accept:

> *Can we predict, at the moment a request is logged, whether it will miss its SLA, so supervisors can intervene early?*

- **Target:** `sla_met` (did it meet the target).
- **Available at logging time:** channel, priority, service target, district, time of day. (Note `resolution_hours` is **not** known at logging time; it is the outcome. Day 3 will treat it carefully to avoid leakage.)
- **Value:** a supervisor who knows a request is at risk can reprioritise it before it slips.

## 8. Communicate the finding
One structured paragraph beats a wall of charts. Use: **what we found, how confident, what to do, what to check next.**

> *Digital channels resolve requests about 15 hours faster at the median than the call centre and walk-in centre, and meet their SLA far more often (around 60% versus 35%). The gap is large and statistically robust (p < 0.001). Low-priority requests miss their SLA most often (only 38% met), suggesting they are deprioritised until they slip. **Recommendation:** review how low-priority and non-digital requests are queued. **To check next:** whether non-digital requests are simply harder cases, rather than worse-handled ones.*

## Your turn
1. Add a feature `hours_since_midnight` or bucket `submitted_hour` into morning / afternoon / evening / night. Which is more useful, and why?
2. One-hot encode `channel_name` with `pd.get_dummies`. How many columns does it add?
3. Rewrite the finding paragraph for a **technical** Day 3 audience instead of a manager. What changes?

In [8]:
# your turn